# SAADE–AUA v5 FINAL · Повний дисертаційний експеримент

**Гіпотеза H1:** online scenario-adaptive навчання (E3) збільшує ASRI відносно випадкової аугментації E2 при тотожній архітектурі, даних, числі епох, seed і вірогідності аугментації 0,5. Нульова гіпотеза: переваги немає. Висновок має випливати з вимірювань, а не з коду.

- E1 Static: без акустичного перетворення.
- E2 Random: рівноймовірні 4 категорії × 5 рівнів.
- E3 Adaptive: періодичне оновлення розподілу 20 впливів на окремій adaptation-частині.
- Validation і hidden test не використовуються для оновлення ваг E3.
- `pilot_existing` лише перевіряє реалізацію; `matched_tts` потребує справжніх REAL–TTS пар. Експертного оцінювання в цьому випуску немає.

Після запуску **посценарні прогнози, аудіоплеєри, осцилограми, спектрограми, ROC, ASRI, bootstrap і зміни адаптивної політики** зберігаються у `results/SAADE_AUA_2026_v5_FINAL`. Ні AUA, ні Melnyk_dus не змінюються.

## 0. Параметри запуску

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys, json, shutil
import numpy as np, pandas as pd
from IPython.display import display

PROJECT=Path('/home/jovyan/work/audio_real_fake')
AUA=PROJECT/'AUA'
# PILOT окремий від справжнього matched-TTS експерименту.
DATA_MODE='pilot_existing'
OUT=PROJECT/'results'/('SAADE_AUA_2026_v5_PILOT' if DATA_MODE=='pilot_existing'
                       else 'SAADE_AUA_2026_v5_FINAL')
assert AUA.is_dir(),f'Не знайдено: {AUA}'
assert shutil.which('ffmpeg'), 'Потрібен системний ffmpeg'

# 1. `pilot_existing` — наявний AUA, ризик конфаундингу джерела;
# 2. `matched_tts` — лише після фактичної генерації з 01_Text_to_TTS.ipynb.
# DATA_MODE задається вище, ДО формування OUT
TTS_RECEIPTS=OUT/'V5_00_PIPER_GENERATION_RECEIPTS.csv'

# Важливо: тут ПОВНЕ навчання УВІМКНЕНО за замовчуванням.
RUN_TRAINING=True
SEEDS=(42,123,777)
MODELS=('spectrogram_cnn','raw_wave_cnn','mfcc_svm')
EPOCHS=15
UPDATE_EVERY=3
MAX_PARENT_FILES=900
UNKNOWN_TRAIN_CAP=150
TARGET_SEGMENTS=1200
REAL_FRACTION=.50
AUGMENTATION_PROBABILITY=.50
ADAPT_MAX_PER_CLASS=24
SHOW_AUDIO_AND_PLOTS=True

assert DATA_MODE in {'pilot_existing','matched_tts'}
assert len(SEEDS)>=3, 'Для повного запуску передбачено 3–5 seeds'
assert 0<REAL_FRACTION<1
print('AUA:',AUA,'\nOUT:',OUT,'\nDATA_MODE:',DATA_MODE,'\nTRAIN:',RUN_TRAINING)

## 1. Імпорти версійно ізольованих модулів v5

In [ ]:
sys.path.insert(0,str(Path.cwd().resolve()))
import saade_final_core as S
import saade_final_data as D
import saade_final_methods as M
import saade_final_visuals as V
import saade_final_adaptive as A
for module in (S,D,M,V,A):
    path=Path(module.__file__).resolve()
    print(module.__name__,path)
    assert path.parent==Path.cwd().resolve(), f'Стороння версія: {path}'
cfg=S.Config(root=str(AUA),out=str(OUT),scenario_profile='v2_5level',seed=42,
             seconds=4.,min_seconds=2.,augmentation_probability=AUGMENTATION_PROBABILITY,
             batch_size=12,lr=0.0005,epochs=EPOCHS,use_cache=True)
assert all(Path(module.__file__).name.startswith('saade_final_') for module in (S,D,M,V,A))

## 2. Каталог AUA за первинними метаданими

In [ ]:
catalog=D.build_verified_catalog(cfg)
print('Аудіозаписів:',len(catalog),'придатних:',catalog.usable.sum())
display(pd.crosstab(catalog.source,catalog.label,dropna=False))
print('DRSSU зв’язані з metadata.csv:',int((catalog.source.eq('DRSSU')&catalog.primary_source_meta_found).sum()))

## 3. Незалежний поділ первинних записів і TRUE Text → TTS

`05_splits` з AUA не використовується повторно як джерело нових звуків. Для справжніх пар перевіряються однаковий текст, SHA-256, voice/model ID і групова незалежність.

In [ ]:
parents,summary=D.prepare_parent_splits(catalog,cfg,max_parent_files=MAX_PARENT_FILES,
             unknown_train_cap=UNKNOWN_TRAIN_CAP,policy='exploratory')
display(summary)
if DATA_MODE=='matched_tts':
    if not TTS_RECEIPTS.is_file():
        raise FileNotFoundError('Немає TTS receipts. Запустіть 01_Text_to_TTS.ipynb та згенеруйте справжнє аудіо.')
    parents=A.integrate_tts_pairs(parents,TTS_RECEIPTS,cfg,min_pairs_per_split=3)
    print('Підтверджених REAL–TTS пар:',parents.pair_id.nunique())
else:
    print('PILOT: це НЕ matched REAL–TTS дані; confounding source↔class зберігається')
    if not (OUT/'V5_00_TTS_GENERATION_QUEUE.csv').exists():
        q=A.generate_tts_job_queue(parents,cfg,max_real=400)
        print('Створено чергу для майбутнього TTS:',len(q))

## 4. Train / Adaptation / Validation / Hidden, неперекривні фрагменти та 50/50

In [ ]:
if DATA_MODE=='matched_tts':
    folds=A.create_matched_adaptation_fold(parents,cfg,seed=42,target_fraction=.22)
else:
    folds=A.create_adaptation_fold(parents,cfg,seed=42,target_fraction=.22)
segments,counts,seg_diag=M.expand_nonoverlapping_segments(
    folds,cfg,target=TARGET_SEGMENTS,max_per_parent=30)
if DATA_MODE=='matched_tts':
    splits=A.balance_pairs(segments,cfg,seed=42)
else:
    splits,balance_report=A.balance_per_partition(segments,cfg,real_pct=REAL_FRACTION,seed=42)
splits.to_csv(OUT/'V5_02_LOCKED_EXPERIMENT_SEGMENTS.csv',index=False)
splits.to_csv(OUT/'03_locked_splits.csv',index=False) # переглядач використовує САМЕ фінальну вибірку
S.lock_hidden_test(splits,cfg)
assert splits.groupby('group_id').split.nunique().max()==1
assert splits.groupby('parent_file_id').split.nunique().max()==1
if DATA_MODE=='matched_tts': assert splits.groupby('pair_id').split.nunique().max()==1
bal=splits.groupby(['split','label']).size().unstack(fill_value=0)
display(bal.rename(columns={0:'REAL',1:'FAKE'}))
assert set(bal.columns)=={0,1}
assert bal[0].eq(bal[1]).all(), 'Не досягнуто 50/50 по всіх частинах'
source_audit,voice_risks=A.source_and_split_diagnostics(splits,cfg)
display(voice_risks)
info,readiness_before=A.research_readiness(splits,cfg,mode=DATA_MODE,trained_seeds=())
display(info)
if DATA_MODE=='matched_tts':
    prerequisites=('independent_groups_ready','independent_tts_voices_across_folds',
                   'no_cross_split_text','no_cross_split_group_or_pair')
    blocked=[p for p in prerequisites if not readiness_before[p]]
    if blocked:
        raise ValueError('Matched experiment is not scientifically ready: '+', '.join(blocked)+
                         '. Provide more REAL/TTS independent groups before training.')
print('Контрольний SHA вибірки:',S.frozen_hash(splits))

## 5. 21 акустичний сценарій, прослуховування й спектрограми

**Сценарії не генерують нові слова** — вони перетворюють сигнал REAL/FAKE. Новий FAKE з тексту створюється лише в Text → TTS ноутбуці.

In [ ]:
guide=V.draw_scenario_guide(cfg,tag='v5_scenario_guide')
display(guide.head(21))
if SHOW_AUDIO_AND_PLOTS:
    examples=V.pick_examples(splits,'validation')
    for r in examples:
        print('REAL' if int(r.label)==0 else 'FAKE',r.relative_path)
        V.show_audio_gallery(r,cfg,tag='v5_audio_gallery',show_players=True,
                             show_figures=True,max_players=None)
    # інтерактивний вибір запису/спотворення, якщо встановлено ipywidgets
    V.make_interactive_player(splits,cfg,'validation')

## 6. Головний експеримент: E1 / E2 / E3, 3 seeds

**Тут дійсно виконується навчання** (за `RUN_TRAINING=True`). E3 оновлює ваги кожні `UPDATE_EVERY` епох на adaptation, а validation читається лише після завершення навчання моделей кожного seed. `hidden_test` не читається.

In [ ]:
validation_predictions=None
if RUN_TRAINING:
    validation_predictions,adaptation_predictions=A.evaluate_all_seeds(
        splits,cfg,models=MODELS,seeds=SEEDS,epochs=EPOCHS,
        update_every=UPDATE_EVERY,adapt_max_per_class=ADAPT_MAX_PER_CLASS)
    print('Завершено E1/E2/E3, seeds:',SEEDS)
    print('Validation predictions:',len(validation_predictions),'рядків')
else:
    print('Навчання вимкнене — наукових метрик немає. Для запуску RUN_TRAINING=True.')

## 7. Повнота сценаріїв, ASRI, EER, ROC-AUC, F1 та головна різниця ΔASRI

In [ ]:
if validation_predictions is not None:
    scenario_metrics,asri_by_seed,paired_deltas,delta_summary=A.metric_tables(validation_predictions,cfg)
    display(asri_by_seed.sort_values(['model','seed','regime']))
    display(paired_deltas)
    display(delta_summary)
    coverage=(validation_predictions.groupby(['seed','model','regime'])
        .agg(n_scenarios=('scenario_id',lambda x:validation_predictions.loc[x.index,['category','level']].drop_duplicates().shape[0]),
             n_predictions=('scenario_id','size')).reset_index())
    coverage.to_csv(OUT/'V5_05_SCENARIO_COVERAGE.csv',index=False)
    display(coverage)
    assert coverage.n_scenarios.eq(21).all(),'Один або кілька сценаріїв пропущено'
    assert (asri_by_seed.complete_cells==20).all(),'Неповна ASRI матриця'
else:
    print('Статистика з’явиться лише після навчання')

## 8. Статистичні тести за незалежними голосами

In [ ]:
if validation_predictions is not None:
    bootstrap=A.stratified_voice_bootstrap(validation_predictions,cfg,
                                            B=500,seed=2026,min_groups_per_class=3)
    display(bootstrap)
    if (bootstrap.status!='OK').any():
        print('НЕДОСТАТНЬО незалежних голосів: не робимо сильних висновків за bootstrap')
    counts=validation_predictions.groupby('label').group_id.nunique()
    if len(counts)==2 and counts.min()>=3:
        for seed,g in validation_predictions.groupby('seed'):
            status=M.interaction_statistics(g,cfg,tag=f'v5_validation_seed{seed}')
            print('GEE interaction, seed',seed,status.get('status'))
    else:
        print('GEE не використовується через брак незалежних REAL/FAKE-груп')

## 9. Візуалізація результатів за всіма 21 сценаріями й адаптивних ваг

In [ ]:
if validation_predictions is not None:
    for _seed,_pred in validation_predictions.groupby('seed'):
        V.report_scenario_tests(_pred,cfg,tag=f'v5_validation_seed{_seed}',show=True)
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(9,4.5))
    for name,g in paired_deltas.groupby('model'):
        ax.plot(g.seed,g.delta_adaptive_minus_random,marker='o',label=name)
    ax.axhline(0,color='black',linewidth=1)
    ax.set(title='Чи допомагає E3? ΔASRI = E3 − E2',xlabel='Випадкове зерно',ylabel='ΔASRI')
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(OUT/'V5_06_ADAPTIVE_EFFECT_BY_SEED.png',dpi=160)
    plt.show()
    for model in MODELS:
        for seed in SEEDS:
            path=OUT/'models'/f'{model}_adaptive_seed{seed}_policy_history.csv'
            if path.exists():
                d=pd.read_csv(path)
                if not d.empty:
                    by=d.drop_duplicates(['epoch','category'])
                    pivot=by.pivot(index='epoch',columns='category',values='category_weight')
                    ax=pivot.plot(marker='o',figsize=(8,3.8),title=f'E3 online weights: {model}, seed={seed}')
                    ax.set(xlabel='Епоха адаптації',ylabel='Імовірність категорії')
                    ax.set_ylim(0,1);ax.grid(alpha=.2)
                    plt.tight_layout();plt.savefig(OUT/f'V5_06_POLICY_{model}_seed{seed}.png',dpi=150);plt.show()
else:
    print('Графіки після навчання')

## 10. Підсумкова готовність та пакування для перевірки

In [ ]:
info,readiness=A.research_readiness(splits,cfg,mode=DATA_MODE,
                                            trained_seeds=(SEEDS if validation_predictions is not None else ()))
print('Готовність до строгого наукового висновку:')
display(pd.DataFrame([readiness]).T.rename(columns={0:'value'}))
if validation_predictions is not None:
    wins=int(paired_deltas.delta_adaptive_minus_random.gt(0).sum())
    print('Кількість позитивних model×seed порівнянь E3−E2:',wins,'/',len(paired_deltas))
    if DATA_MODE!='matched_tts':
        print('ЗАСТЕРЕЖЕННЯ: пілотний AUA має змішування джерела та класу, навіть якщо E3 виграє.')
from zipfile import ZipFile, ZIP_DEFLATED
archive=OUT.parent/'SAADE_v5_FOR_REVIEW.zip'
report_files=[p for p in OUT.rglob('*') if p.is_file() and
              p.suffix.lower() in {'.csv','.json','.png','.txt','.md'} and
              'cache' not in p.parts and 'paired_tts_generated' not in p.parts]
with ZipFile(archive,'w',ZIP_DEFLATED) as z:
    for p in report_files:
        z.write(p,arcname=str(p.relative_to(OUT)))
print('Звіт для перевірки:',archive,'файлів:',len(report_files))
print('Hidden test НЕ відкривався. Для одноразового тесту відкрийте 04_Hidden_Test_ONCE.ipynb.')

---
### Дисертаційна інтерпретація

**Не можна** стверджувати, що H1 доведена лише через позитивне ΔASRI на validation. Потрібні незалежні генератори/голоси, контроль умов запису, багаторазові seeds, достатні групові довірчі інтервали, зафіксований протокол і одноразовий hidden test. Наявність REAL–TTS пар контролює текст, але не обов'язково мікрофонний канал або генераторний стиль. **Експертне аудіювання навмисно не входить до цієї версії.**